In [ ]:
# Library import

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('ticks')

### 1. Dataset Loading

In [ ]:
# Read tracks_clean.csv from notebook 02
tracks = pd.read_csv('../data/tracks_clean.csv')
tracks.head()

### 2. Univariate Analysis

In [ ]:
from scipy import stats

tracks_features = ['popularity', 'danceability', 'energy', 'loudness', 
                   'speechiness', 'acousticness', 'instrumentalness', 
                   'liveness', 'valence', 'tempo']

# --- 1. DESCRIPTIVE STATISTICS ---
# .describe() for count, mean, std, min, quartiles, max
desc_stats = tracks[tracks_features].describe().T
# .skew() and .kurtosis() to integrate the data
desc_stats['skewness'] = tracks[tracks_features].skew()
desc_stats['kurtosis'] = tracks[tracks_features].kurtosis()

# Skewness interpreter
def interpret_skewness(skew_val):
    if abs(skew_val) < 0.5:
        return 'symmetric'
    elif abs(skew_val) < 1:
        return 'moderately_skewed'
    else:
        return 'highly skewed'

desc_stats['skew_label'] = desc_stats['skewness'].apply(interpret_skewness)

desc_stats.round(3)

In [ ]:
# --- 2. HISTOGRAM PLOTS ---
# Create a 5x2 grid of subplots (one per feature)
fig, axes = plt.subplots(5, 2, figsize=(12, 10))

for i, feature in enumerate(tracks_features):
    # Convert linear index i into (row, col) coordinates for the 5x2 grid
    row = i // 2
    col = i % 2
    ax = axes[row, col]

    # Retrieve pre-computed stats
    stats = desc_stats.loc[feature]

    # Draw the histplots
    sns.histplot(
        data=tracks[feature],
        ax=ax,
        bins=50,
        kde=True,
        color=sns.color_palette('viridis', len(tracks_features))[i]
    )

    # Set individual title, remove y tick and x-axis label
    ax.set_title(feature, fontsize=12, fontweight='bold')
    ax.set_xlabel('')      # remove x-axis label
    sns.despine(ax=ax)

fig.suptitle('Distribution of Audio Features', fontsize=16, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

### Key findings include:

- **Popularity** is heavily **right-skewed**, with many tracks having low popularity  
- **Danceability** and **energy** show **simmetric** distributions  
- **Loudness** shows strong **left skew**, reflecting a preference for louder tracks  
- **Speechiness**, **instrumentalness**, and **liveness** exhibit significant **right skew**  
- *Tempo* displays a **multi-modal distribution** consistent with genre differences

### 3. Bivariate Analysis

In [ ]:
# --- 1. CORRELATION HEATMAP ---

# Compute the correlation matrix
corr_matrix = tracks[tracks_features].corr()

fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(
    corr_matrix,         # show lower triangle only
    annot=True,          # print correlation values inside each cell
    fmt=".2f",           # round to 2 decimal places
    cmap='viridis',
    linewidths=0.5,      # thin grid lines between cells
    vmin=-1, vmax=1,     # fix color scale to full correlation range
    ax=ax
)

ax.set_title("Correlation Heatmap of Numerical Features", fontsize=16, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. PAIRWISE CORRELATION | POPULARITY VS AUDIO FEATURES ---

bivariate_cols = [
    'danceability', 'energy', 'loudness', 'valence', 
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'tempo'
]

plt.figure(figsize=(10, 8))
plt.suptitle("Popularity vs Audio Features (Bivariate Analysis)", fontsize=16, fontweight='bold')

sample_tracks = tracks.sample(3000, random_state=42)  # sampling for performance

for i, col in enumerate(bivariate_cols, 1):
    plt.subplot(3, 3, i)
    
    # Calculate Pearson correlation between the feature and popularity
    r = sample_tracks[col].corr(sample_tracks['popularity'])
    
    # Choose color based on threshold
    if r > 0.1:
        color = '#fde725'    # positive correlation
    elif r < -0.1:
        color = '#440256'    # negative correlation
    else:
        color = '#21918c'    # low correlation
    
    sns.scatterplot(data=sample_tracks, x=col, y='popularity', color='#30302e', alpha=0.3)
    sns.regplot(data=sample_tracks, x=col, y='popularity', scatter=False, color=color)
    sns.despine()
    
    # Add r value to the title for transparency
    plt.title(f"popularity vs {col} | r = {r:.2f}", fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

### Key findings include:
- **Danceability** and **energy** show **mild positive** correlation with popularity  
- **Loudness correlates moderately with popularity** due to modern production trends  
- **Valence** (musical positiveness), **speechiness** and **liveness** show very **low correlation with popularity**
- **Acousticness** and **instrumentalness** show **moderate negative correlation** with **popularity** 